> 🔑 **VERSIÓN DOCENTE — no distribuir.**
>
> Tiempos sugeridos: Parte 1 → bloque 2 (08:50–09:25, la generación del dataset grande conviene tenerla hecha o lanzarla al inicio del bloque). Parte 2 → bloque 3 (09:40–10:05, ~8 min por muestra). Parte 3 → 10:05–10:15.
>
> ⚠️ **Este notebook usa 10M filas; el de estudiantes usa 2M.** Es a propósito: 10M × ~25 equipos en las máquinas del laboratorio son ~15 GB de disco y un DataFrame de varios GB por máquina, con riesgo real de que el kernel muera antes del Ejercicio 1.5. La corrida de 10M es la **proyectada**; los equipos ven el mismo efecto con 2M (~120 MB, DataFrame ~400 MB). Si alcanzas a dejar `demo_grande.csv` copiado en cada máquina antes de la clase, mejor todavía.
>
> Respuestas esperadas clave: (1.5) la consulta escala ~lineal pero la carga y la memoria son el cuello; el DataFrame ocupa MÁS que el archivo en disco por los strings de Python; con 70 GB el kernel muere → motiva índices/motores especializados. (A) geometrías no caben bien en columnas escalares; "dentro de la zona" exige operaciones espaciales → PostGIS. (B) esquema variable, desorden temporal y flujo infinito → streaming/NoSQL (Redis, HBase). (C) consultas de N saltos = N self-joins → grafos (Neo4j). No adelantar los nombres de las herramientas a los equipos: la revelación es en la cátedra (diapositiva 14).
>
> Datos verificados (por si algún equipo pregunta): muestra A = 22 features (20 Point + 1 Polygon + 1 LineString). Muestra B = 300 eventos, 19 sin `pm25`, 12 con `-999`, 12 con `bateria_pct`, `ts` desordenado. Muestra C = 15 nodos (8 Persona, 4 Organización, 3 Proyecto) y 30 aristas; **empate de grado 6 entre N07 (Gabriela Pinto) y N14 (Rutas Inteligentes)**, el proyecto top es N14 y sus personas conectadas son Benjamín, Camila, Diego, Emilia y Felipe.

# Laboratorio 01 — Fuentes de datos masivos

**Procesamiento Masivo de Datos (ELE051-B / EIN102B)** · USM 2026-2 · Paralelo 701
**Clase 1** · Viernes 7 de agosto de 2026 · Lab. de Informática 2

---

**Integrantes del equipo:**

1. _(nombre y rol USM)_
2. _(nombre y rol USM)_
3. _(nombre y rol USM)_

## Instrucciones generales

- Trabajen en equipos de 2–3 personas en un solo notebook por equipo.
- Este notebook necesita los archivos de datos que están en la misma carpeta (`Laboratorios/01`): `demo_chico.csv`, `generar_dataset_grande.py`, `muestra_A.geojson`, `muestra_B.ndjson`, `muestra_C_nodos.csv` y `muestra_C_aristas.csv`.
- Las celdas de código que dicen `# Escribe tu código aquí` son suyas: escriban lo necesario para responder la consigna.
- Las celdas que dicen **✍️ Respuesta del equipo** se responden **en texto**, editando la celda.
- Este laboratorio **se trabaja y se cierra en la misma clase — no es un entregable evaluado**. Lo único que debe quedar completo antes de la cátedra (10:15) es la **Parte 3 (Registro por equipo)**, porque la retomamos ahí en conjunto.
- Las actividades evaluadas del curso son las **tareas**, que se enviarán durante el semestre (Tarea 1: 04-sep · Tarea 2: 13-nov).

> ⚠️ Antes de partir: verifiquen que Docker funciona en su equipo (`docker run hello-world` en la terminal). Si falla, avisen **ahora** — todo el semestre depende de esto.

## Configuración

Ejecuten esta celda una vez. Además de las importaciones, define `memoria_pico_mb()`, una función auxiliar que informa la memoria máxima que ha usado este proceso (en MB) — la vamos a usar para medir cuándo el enfoque "cargar todo en RAM" empieza a sufrir.

In [ ]:
import os
import json
import time
import resource
import sys

import pandas as pd
import matplotlib.pyplot as plt

def memoria_pico_mb():
    """Memoria máxima usada por este proceso, en MB."""
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return r / 1e6 if sys.platform == "darwin" else r / 1e3

print("Entorno listo · pandas", pd.__version__)

---
# Parte 1 — La misma consulta, dos datasets

Vamos a ejecutar **exactamente la misma consulta** sobre dos archivos con el mismo esquema:

| Archivo | Filas |
|---|---|
| `demo_chico.csv` | 1.000 |
| `demo_grande.csv` | 10.000.000 (lo generaremos ahora) |

Cada fila es un viaje simulado en el Gran Valparaíso: `id_viaje, zona_origen, zona_destino, inicio, duracion_min, distancia_km`.

**La consulta:** duración promedio de los viajes, agrupada por zona de origen, ordenada de mayor a menor.

### 🎯 Predicción (antes de ejecutar nada)

El dataset grande tiene **10.000 veces** más filas que el chico. Anoten su predicción:

**✍️ Respuesta del equipo — predicción:**

- Tiempo estimado de carga del chico: `___` · del grande: `___`
- Tiempo estimado de la consulta en el chico: `___` · en el grande: `___`
- ¿Alguno se va a caer? ¿Por qué?: `___`

### Ejercicio 1.1 — Dataset chico: carga

Carguen `demo_chico.csv` en un DataFrame **midiendo cuánto demora la carga** (usen `time.perf_counter()` antes y después). Impriman: el tiempo de carga, la cantidad de filas y las primeras 5 filas.

In [ ]:
t0 = time.perf_counter()
df_chico = pd.read_csv("demo_chico.csv")
t_carga_chico = time.perf_counter() - t0

print(f"Tiempo de carga: {t_carga_chico:.4f} s")
print(f"Filas: {len(df_chico):,}")
df_chico.head()

### Ejercicio 1.2 — Dataset chico: la consulta

Calculen la **duración promedio (`duracion_min`) por zona de origen (`zona_origen`)**, ordenada de mayor a menor, midiendo el tiempo de ejecución de la consulta. Muestren el resultado completo (son solo 10 zonas).

In [ ]:
t0 = time.perf_counter()
consulta_chico = (df_chico
                  .groupby("zona_origen")["duracion_min"]
                  .mean()
                  .sort_values(ascending=False))
t_consulta_chico = time.perf_counter() - t0

print(f"Tiempo de consulta: {t_consulta_chico:.4f} s")
consulta_chico.round(1)

### Ejercicio 1.3 — Dataset chico: memoria

Midan cuánta memoria RAM ocupa el DataFrame (pandas puede informarlo, considerando también las columnas de texto) y cuál es la memoria pico del proceso hasta ahora (`memoria_pico_mb()`). Impriman ambos valores en MB.

In [ ]:
mem_df_chico = df_chico.memory_usage(deep=True).sum() / 1e6
print(f"Memoria del DataFrame chico: {mem_df_chico:.1f} MB")
print(f"Memoria pico del proceso:    {memoria_pico_mb():,.0f} MB")

### Generar el dataset grande

Esta celda genera `demo_grande.csv` con el script incluido en la carpeta (si el archivo ya existe, no hace nada). Con 10 millones de filas demora unos minutos y ocupa ~600 MB de disco — mientras corre, comenten sus predicciones.

> Versión docente: 10M es la corrida **proyectada**. El notebook de estudiantes trae `N_FILAS = 2_000_000` para no reventar las máquinas del laboratorio.

In [ ]:
N_FILAS = 10_000_000

if not os.path.exists("demo_grande.csv"):
    import subprocess
    subprocess.run([sys.executable, "generar_dataset_grande.py", str(N_FILAS)], check=True)
else:
    print("demo_grande.csv ya existe, no se regenera.")
print(f"Tamaño en disco: {os.path.getsize('demo_grande.csv') / 1e6:,.0f} MB")

### Ejercicio 1.4 — Dataset grande: repetir todo

Repitan 1.1, 1.2 y 1.3 pero con `demo_grande.csv`: carguen midiendo el tiempo, ejecuten **la misma consulta** midiendo el tiempo, y midan memoria del DataFrame y memoria pico del proceso.

In [ ]:
t0 = time.perf_counter()
df_grande = pd.read_csv("demo_grande.csv")
t_carga_grande = time.perf_counter() - t0
print(f"Tiempo de carga: {t_carga_grande:.2f} s   ({len(df_grande):,} filas)")

t0 = time.perf_counter()
consulta_grande = (df_grande
                   .groupby("zona_origen")["duracion_min"]
                   .mean()
                   .sort_values(ascending=False))
t_consulta_grande = time.perf_counter() - t0
print(f"Tiempo de consulta: {t_consulta_grande:.2f} s")

mem_df_grande = df_grande.memory_usage(deep=True).sum() / 1e6
print(f"Memoria del DataFrame grande: {mem_df_grande:,.0f} MB")
print(f"Memoria pico del proceso:     {memoria_pico_mb():,.0f} MB")
consulta_grande.round(1)

### Ejercicio 1.5 — Tabla comparativa

Completen la tabla con sus mediciones (pueden imprimirla con código o editar esta celda):

| Métrica | chico (1K) | grande (10M) | ¿cuántas veces más? |
|---|---|---|---|
| Tiempo de carga | | | |
| Tiempo de consulta | | | |
| Memoria del DataFrame | | | |

**✍️ Respuesta del equipo:**

1. El archivo creció 10.000×. ¿La carga y la consulta demoraron 10.000× más? ¿Qué escaló peor y qué escaló mejor?
2. La memoria del DataFrame, ¿es mayor o menor que el archivo en disco? ¿Por qué creen que pasa eso?
3. Si el archivo tuviera 1.000 millones de filas (~70 GB), ¿qué pasaría con este enfoque en el computador del laboratorio? ¿Comprar más RAM lo arregla para siempre?

In [ ]:
factor = lambda g, c: f"{g / c:,.0f}x" if c > 0 else "—"
tabla = pd.DataFrame({
    "chico (1K)": [t_carga_chico, t_consulta_chico, mem_df_chico],
    "grande (10M)": [t_carga_grande, t_consulta_grande, mem_df_grande],
}, index=["Tiempo de carga (s)", "Tiempo de consulta (s)", "Memoria DataFrame (MB)"])
tabla["¿cuántas veces más?"] = [
    factor(t_carga_grande, t_carga_chico),
    factor(t_consulta_grande, t_consulta_chico),
    factor(mem_df_grande, mem_df_chico),
]
tabla.round(3)

> 💡 **La lección de la Parte 1:** el problema no es la consulta — es la **estructura**. Cargar todo en RAM funciona hasta que deja de funcionar, y los datos reales no avisan cuándo. De esto se trata el curso.

---
# Parte 2 — Reconocimiento de tres muestras reales

En la carpeta hay tres muestras de datos **de naturaleza distinta**. Su misión: abrirlas, describir qué estructura tiene cada una y detectar **qué se rompe** si intentamos meterlas en una tabla relacional. Todavía no busquen el nombre técnico de cada tipo — eso viene en la cátedra.

## Muestra A — `muestra_A.geojson`

### Ejercicio 2.1 — Explorar la estructura

Abran el archivo con el módulo `json` y respondan con código: ¿cuántos elementos (`features`) tiene? ¿Qué **tipos de geometría** aparecen y cuántos de cada uno? Muestren un elemento completo de ejemplo.

In [ ]:
with open("muestra_A.geojson", encoding="utf-8") as f:
    gj = json.load(f)

features = gj["features"]
print(f"Elementos: {len(features)}")

tipos = pd.Series([ft["geometry"]["type"] for ft in features]).value_counts()
print("\nTipos de geometría:")
print(tipos.to_string())

print("\nEjemplo de elemento:")
features[0]

### Ejercicio 2.2 — Dibujar lo que hay

Dibujen un mapa simple con matplotlib: los **puntos** como un scatter (longitud en x, latitud en y), y encima el resto de las geometrías (la línea como línea, el polígono como contorno cerrado). No necesita ser bonito: necesita mostrar que estos datos **viven en un espacio**.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for ft in features:
    g = ft["geometry"]
    if g["type"] == "Point":
        lon, lat = g["coordinates"]
        ax.scatter(lon, lat, color="steelblue", s=35, zorder=3)
    elif g["type"] == "LineString":
        xs, ys = zip(*g["coordinates"])
        ax.plot(xs, ys, color="darkorange", linewidth=2, label=ft["properties"]["nombre"])
    elif g["type"] == "Polygon":
        xs, ys = zip(*g["coordinates"][0])
        ax.plot(xs, ys, color="firebrick", linewidth=2, label=ft["properties"]["nombre"])

ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_title("Muestra A — ¿qué representa esto?")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

**✍️ Respuesta del equipo — Muestra A:**

1. ¿Qué representa cada registro?
2. ¿Cómo guardarían una geometría (un punto, una ruta, una zona) en una columna de una tabla relacional?
3. ¿Cómo responderían la pregunta *"¿qué estaciones están dentro de la zona de restricción?"* con SQL puro? ¿Qué les falta?

## Muestra B — `muestra_B.ndjson`

Es un archivo **NDJSON**: un objeto JSON por línea. Así llegan típicamente los datos desde un flujo (cada línea = un evento).

### Ejercicio 2.3 — Cargar y describir

Carguen el archivo a un DataFrame (una fila por línea). Respondan con código: ¿cuántos eventos hay? ¿Qué columnas existen y cuántos valores **no nulos** tiene cada una? ¿Cuántas estaciones distintas reportan?

In [ ]:
df_b = pd.read_json("muestra_B.ndjson", lines=True)

print(f"Eventos: {len(df_b)}")
print(f"Estaciones distintas: {df_b['estacion'].nunique()}")
print("\nValores no nulos por columna:")
print(df_b.count().to_string())
df_b.head()

### Ejercicio 2.4 — Encontrar los problemas

Este flujo viene **como llegan los datos en la vida real**. Detecten con código:

1. ¿Los eventos están ordenados por su timestamp (`ts`)? Verifíquenlo, no lo asuman.
2. ¿Cuántos eventos vienen **sin** lectura de `pm25`?
3. ¿Cuántos traen el valor `-999.0` en `pm25`? ¿Qué creen que significa?
4. ¿Cuántos traen el campo `bateria_pct`? ¿Por qué solo algunos?

In [ ]:
df_b["ts"] = pd.to_datetime(df_b["ts"])

ordenado = df_b["ts"].is_monotonic_increasing
print(f"¿Ordenado por tiempo?      {ordenado}")
print(f"Eventos sin pm25:          {df_b['pm25'].isna().sum()}")
print(f"Eventos con pm25 = -999:   {(df_b['pm25'] == -999.0).sum()}")
print(f"Eventos con bateria_pct:   {df_b['bateria_pct'].notna().sum()} de {len(df_b)}")

# Extra docente: eventos por estación
print("\nEventos por estación:")
print(df_b["estacion"].value_counts().sort_index().to_string())

**✍️ Respuesta del equipo — Muestra B:**

1. ¿Qué representa cada registro?
2. En una tabla relacional clásica, ¿qué problemas causan el desorden temporal, los campos faltantes y los campos que solo algunos sensores envían?
3. Este archivo es una foto de un día. Si los sensores mandan eventos **sin parar, para siempre**, ¿qué problema nuevo aparece que no tiene ninguna tabla estática?

## Muestra C — `muestra_C_nodos.csv` + `muestra_C_aristas.csv`

Esta muestra viene en **dos archivos que se usan juntos**.

### Ejercicio 2.5 — Cargar y describir

Carguen ambos archivos. Respondan con código: ¿cuántos nodos hay de cada `tipo`? ¿Cuántas aristas hay de cada `relacion`? Muestren algunas filas de cada archivo para entender cómo se conectan.

In [ ]:
df_nodos = pd.read_csv("muestra_C_nodos.csv")
df_aristas = pd.read_csv("muestra_C_aristas.csv")

print("Nodos por tipo:")
print(df_nodos["tipo"].value_counts().to_string())
print("\nAristas por relación:")
print(df_aristas["relacion"].value_counts().to_string())

display(df_nodos.head(4))
df_aristas.head(6)

### Ejercicio 2.6 — ¿Quién es el más conectado?

Calculen el **grado** de cada nodo (en cuántas aristas participa, como origen o destino) y muestren el top 5 con su etiqueta y tipo. Luego respondan con código una pregunta que obliga a cruzar los dos archivos: ¿qué **personas** están conectadas (por cualquier relación) con el proyecto más conectado?

> Ojo al top 5: puede haber empates en el primer lugar. Fíjense en qué **tipo** de nodo es cada uno.

In [ ]:
grado = (pd.concat([df_aristas["origen"], df_aristas["destino"]])
         .value_counts()
         .rename("grado"))
top = (df_nodos.set_index("id").join(grado).sort_values("grado", ascending=False))
print("Top 5 nodos por grado:")
print(top.head(5).to_string())

# Cruce entre los dos archivos: aristas -> ids vecinos -> nodos, filtrando por tipo
proyecto_top = top[top["tipo"] == "Proyecto"].index[0]
nombre_proyecto = df_nodos.set_index("id").loc[proyecto_top, "etiqueta"]
conectados = df_aristas.query("origen == @proyecto_top or destino == @proyecto_top")
ids_vecinos = (set(conectados["origen"]) | set(conectados["destino"])) - {proyecto_top}
personas = df_nodos[(df_nodos["id"].isin(ids_vecinos)) & (df_nodos["tipo"] == "Persona")]
print(f"\nPersonas conectadas con '{nombre_proyecto}':")
print(personas["etiqueta"].to_string(index=False))

**✍️ Respuesta del equipo — Muestra C:**

1. ¿Qué representa cada registro en cada uno de los dos archivos? ¿Dónde está la información "importante": en los nodos o en las conexiones?
2. El Ejercicio 2.6 necesitó un *join* implícito entre nodos y aristas. ¿Cómo sería responder *"los colaboradores de los colaboradores de Antonia"* (2 saltos por la relación `COLABORA_CON`) con SQL? ¿Y si fueran 6 saltos?
3. ¿Qué se pierde o se vuelve doloroso al guardar estos datos en tablas relacionales?

---
# Parte 3 — Registro por equipo (cierre de la sesión)

Sinteticen lo trabajado. Editen esta celda y completen la tabla **con sus palabras** — debe quedar lista antes del bloque de cátedra (10:15), porque la retomaremos ahí en conjunto. No se entrega ni se califica: es su material de trabajo para la discusión.

| | Muestra A | Muestra B | Muestra C |
|---|---|---|---|
| ¿Qué representa cada registro? | | | |
| ¿Cómo se relacionan los registros entre sí? | | | |
| ¿Cómo la guardarían en una tabla relacional? | | | |
| ¿Qué se rompe o se pierde al hacerlo? | | | |
| Hipótesis: ¿qué tipo de base de datos necesitaría? | | | |

**Pregunta final del equipo:** de las tres muestras, ¿cuál creen que es la más difícil de manejar a gran escala, y por qué?

**✍️ Respuesta:** `___`

---
*Fin del Laboratorio 01. En la cátedra de hoy (10:15) le ponemos nombre a lo que acaban de descubrir — y en la próxima clase (14-ago) se presenta el proyecto semestral.*